In [9]:
from pydantic import BaseModel, Field
from typing import Optional, List

class SupportTicket(BaseModel):
    summary: str = Field(description="A concise 1-sentence summary of the user's issue.")
    intent: str = Field(description="Must be one of: 'REFUND', 'TECHNICAL_ISSUE', 'ACCOUNT_LOCKED', 'OTHER'")
    urgency: str = Field(description="Must be: 'HIGH', 'MEDIUM', or 'LOW'")
    order_id: Optional[str] = Field(default=None, description="The alphanumeric order ID if mentioned, e.g., ORD-1234")
    frustration_flags: List[str] = Field(default_factory=list, description="Quotes from the user showing anger or legal threats")

ticket = SupportTicket.model_json_schema()


In [10]:
SYSTEM_PROMPT = """
You are a precision data extraction pipeline for a customer support system.
Your job is to extract data from messy user email into a strict JSON format.  

RULES:
1. Do not invent information. If an order ID is missing, keep it null.
2. Only classify intent into the approved categories.
3. Pay close attention to the user's tone to identify the frustration and concern.

Hey, I bought a toaster last week (Order #TX-992) and it literally caught fire. I want my money back NOW.
{"summary": "Toaster caught fire after a week of use.", "intent": "REFUND", "urgency": "HIGH", "order_id": "TX-992", "frustration_flags": ["I want my money back NOW."]}

Hi team, how do I reset my password? I clicked the link but it says expired.
{"summary": "User unable to reset password due to expired link.", "intent": "ACCOUNT_LOCKED", "urgency": "LOW", "order_id": null, "frustration_flags": []}
"""

In [11]:
import os
from groq import Groq
from dotenv import load_dotenv
import json

load_dotenv()

True

In [12]:
API_KEY = os.getenv("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("API KEY NOT FOUND")

In [13]:
client = Groq(api_key=API_KEY)

In [14]:
def extract(text: str) -> SupportTicket:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Extract data from this email:\n\n{text}\n"}
    ]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0
    )

    raw_json = response.choices[0].message.content
    return SupportTicket.model_validate_json(raw_json)

In [15]:
test_emails = [
    {
        "id": "test_01_clean",
        "text": "Hi Support Team, I received my mechanical keyboard today (Order ID: ORD-88192)...",
        "expected_intent": "REFUND",
        "expected_order_id": "ORD-88192"
    },
    {
        "id": "test_03_legal_threat",
        "text": "If my money is not refunded to my card within 24 hours, I am contacting my lawyer...",
        "expected_urgency": "HIGH"
    }
]

In [19]:
import time

for email in test_emails:
    print(f"\nProcessing: {email['id']}")
    answer = extract(email["text"]) 
    
    print(answer.model_dump_json(indent=2))
    time.sleep(5)


Processing: test_01_clean
{
  "summary": "User received mechanical keyboard",
  "intent": "ORDER_FOLLOWUP",
  "urgency": "LOW",
  "order_id": "ORD-88192",
  "frustration_flags": []
}

Processing: test_03_legal_threat
{
  "summary": "User demands refund within 24 hours",
  "intent": "REFUND",
  "urgency": "HIGH",
  "order_id": null,
  "frustration_flags": [
    "I am contacting my lawyer"
  ]
}
